# First Import — RAW_SHAPEFILE → PREPARE_GDB

This notebook implements **First_Import_Rule.md** for the first acquisition cycle.

Primary workflow:

```text
00_raw\<ACQUIRED>
    ↓ immutable source

01_acquired\A<ACQUIRED>.gdb
    ↓ normalized staging feature classes

A<BRANCH_CODE>_47
A<BRANCH_CODE>_48
```

The notebook intentionally does **not** create:

- `APID`
- `INSIDE_X`
- `INSIDE_Y`
- `ATTR_HASH`
- `GEOM_HASH`
- `ENTITY_ID`
- `VERSION_ID`
- `RECORD_STATUS`

Those belong to later validation/archive stages.

## Execution environment

Run this notebook with the **ArcGIS Pro Python environment** because it uses `arcpy`.

The `OldBranch` lookup is read from Excel with ArcGIS `ExcelToTable`. Microsoft Excel drivers therefore need to be available to ArcGIS Pro.

## Important rebuild behavior

`REBUILD_PREPARE_GDB = True` means an existing `A<ACQUIRED>.gdb` is deleted and rebuilt from immutable RAW data. A safety check restricts deletion to the configured `01_acquired` directory and a geodatabase whose name starts with `A`.

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import datetime as dt
from decimal import Decimal, InvalidOperation
import re
import unicodedata
import uuid
import traceback
import hashlib

try:
    import arcpy
except ImportError as exc:
    raise RuntimeError(
        "This notebook must be run with the ArcGIS Pro Python environment "
        "because it requires arcpy."
    ) from exc

arcpy.env.overwriteOutput = True

# ---------------------------------------------------------------------
# Acquisition configuration
# ---------------------------------------------------------------------

ACQUIRED = "2022-05"

BASE_DIR = Path(r"D:\A02-Projects\Refine")

RAW_SHAPEFILE = BASE_DIR / "00_raw" / ACQUIRED
PREPARE_GDB = BASE_DIR / "01_acquired" / f"A{ACQUIRED}.gdb"

REPORT_DIR = BASE_DIR / "05_reports" / ACQUIRED
ACQUIRED_REPORT = REPORT_DIR / f"01_{ACQUIRED}-Import.xlsx"

BRANCH_LOOKUP = Path(r"D:\A03-Resources\Infomation\API\IPADDRESS.xlsx")
BRANCH_LOOKUP_SHEET = "OldBranch"

REFERENCE_GDB = BASE_DIR / "02_ref" / "district.gdb"

# Temporary workspace. Deleted after a successful run unless KEEP_WORK_GDB=True.
WORK_GDB = REPORT_DIR / f"_first_import_work_{ACQUIRED.replace('-', '_')}.gdb"

SUPPORTED_WKID = {
    24047: "47",
    24048: "48",
}

# First-cycle initialization sentinel defined by First_Import_Rule.md.
BASELINE_LAST_UPD_D = "20220531000000"

# Resume is the default for large acquisition cycles.
# False = keep PREPARE_GDB and continue from persistent checkpoints.
# True  = intentionally delete PREPARE_GDB and rebuild everything.
REBUILD_PREPARE_GDB = False

# Skip source shapefiles already marked COMPLETE when their source
# signature is unchanged. Interrupted files are rolled back and retried.
RESUME_IMPORT = True

# Set True while debugging if you want to inspect temporary copied feature classes/tables.
KEEP_WORK_GDB = False

# Maximum number of distinct source/resolved IDs shown in one FILES report cell.
MAX_DISTINCT_PREVIEW = 20

print("ACQUIRED       :", ACQUIRED)
print("RAW_SHAPEFILE :", RAW_SHAPEFILE)
print("PREPARE_GDB   :", PREPARE_GDB)
print("REPORT        :", ACQUIRED_REPORT)
print("REFERENCE_GDB :", REFERENCE_GDB)


## Implementation decisions

The notebook applies the rule conservatively:

1. RAW shapefiles are never edited.
2. Only polygon shapefiles with WKID `24047` or `24048` are routed.
3. Routing is performed **by row** after normalization. If one shapefile contains multiple valid `BRANCH_CODE` values, rows are split into the appropriate target feature classes.
4. `BRANCH_CODE` supplied by the source is preserved when valid. `OldBranch` is used to validate or fill missing information, not to replace history with the current `Active` branch.
5. `LANDOFFICE`-only rows are blocked when one `LANDOFFICE` maps to several historical branch codes.
6. A valid source `BRANCH_CODE` can still be routed when `LANDOFFICE` cannot be filled; the LANDOFFICE problem is reported as a data issue for later review.
7. Known aliases are normalized to canonical staging names. Unmapped source fields remain.
8. If multiple aliases for the same canonical field conflict, the canonical value follows explicit alias precedence and the conflicting original alias fields are retained in staging so no source information is lost.
9. `LAST_UPD_D` is **not required to exist in the source shapefile**. If the field is absent, the notebook creates the canonical staging field and fills `20220531000000`. If the field exists, its source values are preserved and validated.
10. The import is resumable. `PREPARE_GDB` is retained by default and persistent control tables record completed and interrupted source files.
11. A source file marked `COMPLETE` with the same source signature is skipped on the next run.
12. If execution stops during one shapefile, the next run rolls back only that incomplete shapefile to its pre-append target OID/count checkpoints and retries it. Completed shapefiles are retained.
13. Reference candidates and report metadata are checkpointed so resumed runs can still build a complete report.


In [ ]:
# ---------------------------------------------------------------------
# Canonical staging schema
# ---------------------------------------------------------------------

# ArcGIS field type, field length (None for numeric fields)
CANONICAL_SPECS = {
    "CHANGWAT_CODE": ("TEXT", 2),
    "CHANGWAT_NAME": ("TEXT", 255),
    "AMPHOE_CODE": ("TEXT", 4),
    "AMPHOE_NAME": ("TEXT", 255),
    "TAMBON_CODE": ("TEXT", 6),
    "TAMBON_NAME": ("TEXT", 255),
    "LANDOFFICE": ("LONG", None),
    "BRANCH_CODE": ("TEXT", 8),
    "BRANCH_NAME": ("TEXT", 255),
    "UTM1": ("TEXT", 4),
    "UTM2": ("LONG", None),
    "UTM3": ("TEXT", 4),
    "UTM4": ("TEXT", 3),
    "UTMSCALE": ("LONG", None),
    "LAND_NO": ("TEXT", 20),
    "PARCEL_TYPE": ("LONG", None),
    "PARCEL_NAME": ("TEXT", 255),
    "DOC_TYPE": ("LONG", None),
    "LAST_UPD_D": ("TEXT", 14),
    "REMARK": ("TEXT", 250),
}

# Ordered aliases = precedence when more than one alias is present.
# Canonical field is placed first where it may already exist in some source data.
FIELD_ALIASES = {
    "CHANGWAT_CODE": ["CHANGWAT_CODE", "PROV_CODE"],
    "CHANGWAT_NAME": ["CHANGWAT_NAME", "PROV_NAM_T"],
    "AMPHOE_CODE": ["AMPHOE_CODE", "AMP_CODE"],
    "AMPHOE_NAME": ["AMPHOE_NAME", "AMPHOE_T"],
    "TAMBON_CODE": ["TAMBON_CODE", "TAM_CODE"],
    "TAMBON_NAME": ["TAMBON_NAME", "TAM_NAM_T"],
    "LANDOFFICE": ["LANDOFFICE"],
    "BRANCH_CODE": ["BRANCH_CODE", "ORG_ID"],
    "BRANCH_NAME": ["BRANCH_NAME", "ORG_SEQ"],
    "UTM1": ["UTM1", "UTMMAP1"],
    "UTM2": ["UTM2", "UTMMAP2"],
    "UTM3": ["UTM3", "UTMMAP3"],
    "UTM4": ["UTM4", "UTMMAP4"],
    "UTMSCALE": ["UTMSCALE"],
    "LAND_NO": ["LAND_NO"],
    "PARCEL_TYPE": ["PARCEL_TYPE", "PARCEL_TYP"],
    "PARCEL_NAME": ["PARCEL_NAME", "PARCEL_NAM", "PARCELDESC"],
    "DOC_TYPE": ["DOC_TYPE"],
    "LAST_UPD_D": ["LAST_UPD_D"],
    "REMARK": ["REMARK", "REMARKS", "LAND_NAME", "REMARK_ZON"],
}

# These fields must exist in staging even if no source alias exists.
REQUIRED_STAGE_FIELDS = {
    "BRANCH_CODE",
    "LANDOFFICE",
    "LAST_UPD_D",
}

REFERENCE_DEFINITIONS = {
    "CHANGWAT": ("CHANGWAT_CODE", "CHANGWAT_NAME"),
    "AMPHOE": ("AMPHOE_CODE", "AMPHOE_NAME"),
    "TAMBON": ("TAMBON_CODE", "TAMBON_NAME"),
    "PARCEL": ("PARCEL_TYPE", "PARCEL_NAME"),
}

CONTROLLED_UTMSCALES = {500, 1000, 2000, 4000, 5000}

SYSTEM_FIELD_NAMES = {
    "OBJECTID", "FID", "OID",
    "SHAPE", "SHAPE_LENGTH", "SHAPE_AREA",
}

# Reverse alias lookup for the FIELDS report.
ALIAS_TO_CANONICAL = {}
for canonical_name, aliases in FIELD_ALIASES.items():
    for alias in aliases:
        ALIAS_TO_CANONICAL[alias.upper()] = canonical_name


class IssueBook:
    """Aggregate report issues instead of writing millions of record-level rows."""

    def __init__(self):
        self._items = {}

    def add(
        self,
        source_path,
        severity,
        issue_code,
        message,
        count=1,
        sample_oid=None,
    ):
        key = (
            str(source_path),
            str(severity),
            str(issue_code),
            str(message),
        )
        if key not in self._items:
            self._items[key] = {
                "SOURCE_PATH": str(source_path),
                "SEVERITY": str(severity),
                "ISSUE_CODE": str(issue_code),
                "MESSAGE": str(message),
                "ISSUE_COUNT": 0,
                "SAMPLE_OID": None,
            }

        item = self._items[key]
        item["ISSUE_COUNT"] += int(count)

        if item["SAMPLE_OID"] is None and sample_oid is not None:
            item["SAMPLE_OID"] = str(sample_oid)

    def rows(self):
        return list(self._items.values())

    def count_for(self, source_path):
        source_path = str(source_path)
        return sum(
            row["ISSUE_COUNT"]
            for row in self._items.values()
            if row["SOURCE_PATH"] == source_path
        )


def normalize_text(value):
    """
    Normalize a scalar into trimmed Unicode text.
    Empty text becomes None.

    Integer-like floats are serialized without '.0' so Excel-derived
    branch codes such as 48060000.0 become '48060000'.
    """
    if value is None:
        return None

    if isinstance(value, float) and value.is_integer():
        value = int(value)

    text = unicodedata.normalize("NFKC", str(value)).strip()
    return text if text else None


def to_long(value):
    """Convert integer-like values to Python int without silent rounding."""
    if value is None:
        return None

    if isinstance(value, bool):
        return int(value)

    if isinstance(value, int):
        return value

    if isinstance(value, float):
        if value.is_integer():
            return int(value)
        raise ValueError(f"Non-integral numeric value: {value!r}")

    text = normalize_text(value)
    if text is None:
        return None

    try:
        number = Decimal(text)
    except InvalidOperation as exc:
        raise ValueError(f"Not an integer value: {value!r}") from exc

    if number != number.to_integral_value():
        raise ValueError(f"Non-integral numeric value: {value!r}")

    return int(number)


def convert_canonical_value(canonical_field, value):
    """Convert one source value to the canonical staging type."""
    field_type, _ = CANONICAL_SPECS[canonical_field]

    if field_type == "TEXT":
        return normalize_text(value)

    if field_type == "LONG":
        return to_long(value)

    raise ValueError(f"Unsupported canonical field type: {field_type}")


def valid_last_upd_d(value):
    if value is None:
        return True
    text = normalize_text(value)
    if text is None or len(text) != 14 or not text.isdigit():
        return False
    try:
        dt.datetime.strptime(text, "%Y%m%d%H%M%S")
        return True
    except ValueError:
        return False


def valid_branch_code(value):
    value = normalize_text(value)
    return bool(value and re.fullmatch(r"\d{8}", value))


def valid_fixed_digit_text(value, length):
    value = normalize_text(value)
    return bool(value and re.fullmatch(rf"\d{{{length}}}", value))


def field_map(dataset):
    return {field.name.upper(): field for field in arcpy.ListFields(str(dataset))}


def get_actual_field(dataset, wanted_name):
    return field_map(dataset).get(wanted_name.upper())


def is_system_field(field):
    return (
        field.required
        or field.type in {"OID", "Geometry"}
        or field.name.upper() in SYSTEM_FIELD_NAMES
    )


def unique_field_name(dataset, preferred):
    existing = {f.name.upper() for f in arcpy.ListFields(str(dataset))}
    candidate = preferred[:64]

    if candidate.upper() not in existing:
        return candidate

    n = 1
    while True:
        suffix = f"_{n}"
        candidate = f"{preferred[:64-len(suffix)]}{suffix}"
        if candidate.upper() not in existing:
            return candidate
        n += 1


def add_field_from_spec(dataset, field_name, spec):
    field_type, length = spec

    kwargs = {}
    if field_type == "TEXT":
        kwargs["field_length"] = int(length)

    arcpy.management.AddField(
        str(dataset),
        field_name,
        field_type,
        **kwargs,
    )


def preview_values(values, limit=MAX_DISTINCT_PREVIEW):
    cleaned = sorted(
        {str(v) for v in values if v is not None and str(v) != ""}
    )
    if len(cleaned) <= limit:
        return ", ".join(cleaned)
    return ", ".join(cleaned[:limit]) + f" ... (+{len(cleaned)-limit} more)"


def arc_field_type_for_add(field):
    mapping = {
        "String": "TEXT",
        "SmallInteger": "SHORT",
        "Integer": "LONG",
        "Single": "FLOAT",
        "Double": "DOUBLE",
        "Date": "DATE",
        "GUID": "GUID",
    }
    return mapping.get(field.type)


def expected_arc_field_class(spec):
    """Convert AddField type into ListFields type family for comparisons."""
    field_type, _ = spec
    return {
        "TEXT": "String",
        "SHORT": "SmallInteger",
        "LONG": "Integer",
        "FLOAT": "Single",
        "DOUBLE": "Double",
        "DATE": "Date",
        "GUID": "GUID",
    }[field_type]


In [ ]:
# ---------------------------------------------------------------------
# Workspace setup, lookup loading, and source inventory
# ---------------------------------------------------------------------

def create_file_gdb(path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    arcpy.management.CreateFileGDB(str(path.parent), path.name)
    return path


def setup_workspaces():
    if not RAW_SHAPEFILE.exists():
        raise FileNotFoundError(f"RAW_SHAPEFILE does not exist: {RAW_SHAPEFILE}")

    if not BRANCH_LOOKUP.exists():
        raise FileNotFoundError(f"BRANCH_LOOKUP does not exist: {BRANCH_LOOKUP}")

    REPORT_DIR.mkdir(parents=True, exist_ok=True)
    PREPARE_GDB.parent.mkdir(parents=True, exist_ok=True)
    REFERENCE_GDB.parent.mkdir(parents=True, exist_ok=True)

    # Temporary workspace can always be recreated because it contains no
    # persistent resume state.
    if arcpy.Exists(str(WORK_GDB)):
        arcpy.management.Delete(str(WORK_GDB))
    create_file_gdb(WORK_GDB)

    if arcpy.Exists(str(PREPARE_GDB)):
        if REBUILD_PREPARE_GDB:
            safe_parent = PREPARE_GDB.parent.name.lower() == "01_acquired"
            safe_name = (
                PREPARE_GDB.name.startswith("A")
                and PREPARE_GDB.suffix.lower() == ".gdb"
            )

            if not (safe_parent and safe_name):
                raise RuntimeError(
                    "Safety check refused to delete PREPARE_GDB because the "
                    "configured path does not look like "
                    "01_acquired\\A<ACQUIRED>.gdb"
                )

            print(f"REBUILD requested: deleting {PREPARE_GDB}")
            arcpy.management.Delete(str(PREPARE_GDB))
            create_file_gdb(PREPARE_GDB)
        else:
            print(f"RESUME mode: keeping existing {PREPARE_GDB}")
    else:
        create_file_gdb(PREPARE_GDB)

    # district.gdb is persistent: create only if it does not exist.
    if not arcpy.Exists(str(REFERENCE_GDB)):
        create_file_gdb(REFERENCE_GDB)


def load_oldbranch_lookup(issue_book):
    """
    Import OldBranch into WORK_GDB, then build:
      - valid_pairs: (BRANCH_CODE, LANDOFFICE)
      - branch_to_landoffices
      - landoffice_to_branches

    STATUS is deliberately not used to auto-select a branch.
    """
    lookup_table = WORK_GDB / "OldBranch"

    try:
        arcpy.conversion.ExcelToTable(
            str(BRANCH_LOOKUP),
            str(lookup_table),
            BRANCH_LOOKUP_SHEET,
        )
    except Exception as exc:
        raise RuntimeError(
            "ArcGIS could not read the OldBranch Excel sheet. "
            "Check the workbook path, sheet name, and ArcGIS Excel driver installation."
        ) from exc

    fields = field_map(lookup_table)

    branch_field = fields.get("BRANCHCODE")
    landoffice_field = fields.get("LANDOFFICE")
    status_field = fields.get("STATUS")

    if branch_field is None or landoffice_field is None:
        raise RuntimeError(
            "OldBranch must contain BRANCHCODE and LANDOFFICE columns."
        )

    cursor_fields = [branch_field.name, landoffice_field.name]
    if status_field is not None:
        cursor_fields.append(status_field.name)

    valid_pairs = set()
    branch_to_landoffices = defaultdict(set)
    landoffice_to_branches = defaultdict(set)

    invalid_lookup_rows = 0

    with arcpy.da.SearchCursor(str(lookup_table), cursor_fields) as cursor:
        for row in cursor:
            branch = normalize_text(row[0])

            try:
                landoffice = to_long(row[1])
            except ValueError:
                landoffice = None

            if not valid_branch_code(branch) or landoffice is None:
                invalid_lookup_rows += 1
                continue

            valid_pairs.add((branch, landoffice))
            branch_to_landoffices[branch].add(landoffice)
            landoffice_to_branches[landoffice].add(branch)

    if invalid_lookup_rows:
        issue_book.add(
            BRANCH_LOOKUP,
            "DATA ISSUE",
            "INVALID_OLDBRANCH_ROW",
            "OldBranch contains rows with invalid BRANCHCODE or LANDOFFICE.",
            count=invalid_lookup_rows,
        )

    return {
        "valid_pairs": valid_pairs,
        "branch_to_landoffices": branch_to_landoffices,
        "landoffice_to_branches": landoffice_to_branches,
    }


def discover_shapefiles():
    shapefiles = [
        path for path in RAW_SHAPEFILE.rglob("*")
        if path.is_file() and path.suffix.lower() == ".shp"
    ]
    return sorted(shapefiles, key=lambda p: str(p).lower())


def field_inventory_rows(source_path):
    rows = []

    for field in arcpy.ListFields(str(source_path)):
        canonical = ALIAS_TO_CANONICAL.get(field.name.upper())

        if is_system_field(field):
            action = "SYSTEM"
            canonical_out = field.name
        elif canonical is None:
            action = "KEEP"
            canonical_out = field.name
        else:
            canonical_out = canonical
            spec = CANONICAL_SPECS[canonical]
            expected_type = expected_arc_field_class(spec)

            if field.name.upper() != canonical.upper():
                action = "NORMALIZE_NAME"
            elif field.type != expected_type:
                action = "NORMALIZE_TYPE"
            else:
                action = "NORMALIZE_VALUE"

        rows.append({
            "SOURCE_PATH": str(source_path),
            "SOURCE_FIELD": field.name,
            "SOURCE_TYPE": field.type,
            "SOURCE_LENGTH": field.length if field.type == "String" else None,
            "CANONICAL_FIELD": canonical_out,
            "ACTION": action,
        })

    return rows


def dataset_wkid(dataset):
    sr = arcpy.Describe(str(dataset)).spatialReference

    for attr_name in ("factoryCode", "PCSCode"):
        value = getattr(sr, attr_name, None)
        if value not in (None, 0):
            return int(value)

    return 0


def dataset_geometry_type(dataset):
    return str(arcpy.Describe(str(dataset)).shapeType)


In [ ]:
# ---------------------------------------------------------------------
# Per-source normalization and routing preparation
# ---------------------------------------------------------------------

def validate_utm_values(values, issue_book, source_path, oid):
    utm1 = values.get("UTM1")
    utm2 = values.get("UTM2")
    utm3 = values.get("UTM3")
    utm4 = values.get("UTM4")
    scale = values.get("UTMSCALE")

    if utm1 is not None and not valid_fixed_digit_text(utm1, 4):
        issue_book.add(
            source_path, "DATA ISSUE", "INVALID_UTM1",
            "UTM1 must be 4 digits when supplied.",
            sample_oid=oid,
        )

    if utm2 is not None and utm2 not in {1, 2, 3, 4}:
        issue_book.add(
            source_path, "DATA ISSUE", "INVALID_UTM2",
            "UTM2 must be an integer from 1 to 4.",
            sample_oid=oid,
        )

    if utm3 is not None and not valid_fixed_digit_text(utm3, 4):
        issue_book.add(
            source_path, "DATA ISSUE", "INVALID_UTM3",
            "UTM3 must be 4 digits when supplied.",
            sample_oid=oid,
        )

    if utm4 is not None:
        text = normalize_text(utm4)
        if not text or not re.fullmatch(r"\d{2,3}", text):
            issue_book.add(
                source_path, "DATA ISSUE", "INVALID_UTM4",
                "UTM4 must contain 2 or 3 digits when supplied.",
                sample_oid=oid,
            )

    if scale is not None and scale not in CONTROLLED_UTMSCALES:
        issue_book.add(
            source_path, "DATA ISSUE", "INVALID_UTMSCALE",
            "UTMSCALE is outside the controlled values 500/1000/2000/4000/5000.",
            sample_oid=oid,
        )

    if scale == 5000 and utm3 is not None and normalize_text(utm3) != "0000":
        issue_book.add(
            source_path, "DATA ISSUE", "UTM3_SCALE_CONFLICT",
            'UTMSCALE=5000 requires UTM3="0000".',
            sample_oid=oid,
        )

    if scale is not None and utm4 is not None:
        u4 = normalize_text(utm4)

        valid_by_scale = True
        if scale == 5000:
            valid_by_scale = bool(u4 and re.fullmatch(r"\d{3}", u4))
        elif scale == 4000:
            valid_by_scale = (u4 == "00")
        elif scale == 2000:
            valid_by_scale = u4 in {"01", "02", "03", "04"}
        elif scale == 1000:
            valid_by_scale = bool(
                u4 and re.fullmatch(r"\d{2}", u4) and 1 <= int(u4) <= 16
            )
        elif scale == 500:
            valid_by_scale = bool(
                u4 and re.fullmatch(r"\d{2}", u4) and 1 <= int(u4) <= 64
            )

        if not valid_by_scale:
            issue_book.add(
                source_path, "DATA ISSUE", "UTM4_SCALE_CONFLICT",
                "UTM4 is inconsistent with UTMSCALE according to the canonical rule.",
                sample_oid=oid,
            )


def validate_admin_hierarchy(values, issue_book, source_path, oid):
    changwat = values.get("CHANGWAT_CODE")
    amphoe = values.get("AMPHOE_CODE")
    tambon = values.get("TAMBON_CODE")
    branch = values.get("BRANCH_CODE")

    if changwat is not None and not valid_fixed_digit_text(changwat, 2):
        issue_book.add(
            source_path, "DATA ISSUE", "INVALID_CHANGWAT_CODE",
            "CHANGWAT_CODE must be 2 digits.",
            sample_oid=oid,
        )

    if amphoe is not None and not valid_fixed_digit_text(amphoe, 4):
        issue_book.add(
            source_path, "DATA ISSUE", "INVALID_AMPHOE_CODE",
            "AMPHOE_CODE must be 4 digits.",
            sample_oid=oid,
        )

    if tambon is not None and not valid_fixed_digit_text(tambon, 6):
        issue_book.add(
            source_path, "DATA ISSUE", "INVALID_TAMBON_CODE",
            "TAMBON_CODE must be 6 digits.",
            sample_oid=oid,
        )

    if changwat and amphoe and not amphoe.startswith(changwat):
        issue_book.add(
            source_path, "DATA ISSUE", "CHANGWAT_AMPHOE_CONFLICT",
            "AMPHOE_CODE does not start with CHANGWAT_CODE.",
            sample_oid=oid,
        )

    if changwat and tambon and not tambon.startswith(changwat):
        issue_book.add(
            source_path, "DATA ISSUE", "CHANGWAT_TAMBON_CONFLICT",
            "TAMBON_CODE does not start with CHANGWAT_CODE.",
            sample_oid=oid,
        )

    if amphoe and tambon and not tambon.startswith(amphoe):
        issue_book.add(
            source_path, "DATA ISSUE", "AMPHOE_TAMBON_CONFLICT",
            "TAMBON_CODE does not start with AMPHOE_CODE.",
            sample_oid=oid,
        )

    if changwat and branch and valid_branch_code(branch) and not branch.startswith(changwat):
        issue_book.add(
            source_path, "DATA ISSUE", "CHANGWAT_BRANCH_CONFLICT",
            "BRANCH_CODE does not start with CHANGWAT_CODE.",
            sample_oid=oid,
        )


def collect_reference_candidate(reference_candidates, values):
    for table_name, (code_field, name_field) in REFERENCE_DEFINITIONS.items():
        code = values.get(code_field)
        name = values.get(name_field)

        if code is None or name is None:
            continue

        name = normalize_text(name)
        if name is not None:
            reference_candidates[table_name].add((code, name))


def normalize_staging_feature_class(
    temp_fc,
    source_path,
    source_count,
    lookup,
    issue_book,
    reference_candidates,
):
    """
    Normalize all known aliases in one UpdateCursor pass.

    Returns row-level routing statistics without adding provenance/helper
    fields to the final staging schema.
    """
    initial_fields = field_map(temp_fc)

    present_aliases = {}
    for canonical, aliases in FIELD_ALIASES.items():
        actual = []
        for alias in aliases:
            field = initial_fields.get(alias.upper())
            if field is not None:
                actual.append(field.name)
        present_aliases[canonical] = actual

    canonical_to_build = [
        canonical
        for canonical in FIELD_ALIASES
        if present_aliases[canonical] or canonical in REQUIRED_STAGE_FIELDS
    ]

    # Temporary canonical fields guarantee the correct GDB type even when
    # the source alias has an incompatible shapefile type.
    temp_dest_fields = {}
    for canonical in canonical_to_build:
        temp_name = unique_field_name(temp_fc, f"N_{canonical}")
        add_field_from_spec(temp_fc, temp_name, CANONICAL_SPECS[canonical])
        temp_dest_fields[canonical] = temp_name

    source_cursor_fields = []
    seen = set()
    for canonical in canonical_to_build:
        for actual in present_aliases[canonical]:
            if actual.upper() not in seen:
                source_cursor_fields.append(actual)
                seen.add(actual.upper())

    cursor_fields = ["OID@"] + source_cursor_fields + list(temp_dest_fields.values())
    index = {name.upper(): i for i, name in enumerate(cursor_fields) if name != "OID@"}
    oid_index = 0

    branch_counts = Counter()
    blocked_count = 0

    source_branch_values = set()
    source_landoffice_values = set()
    resolved_branch_values = set()
    resolved_landoffice_values = set()

    canonical_conflicts = set()

    source_has_last_upd_d = bool(present_aliases["LAST_UPD_D"])

    with arcpy.da.UpdateCursor(str(temp_fc), cursor_fields) as cursor:
        for row in cursor:
            oid = row[oid_index]
            values = {}

            # ---------------------------------------------------------
            # Alias normalization + explicit type conversion
            # ---------------------------------------------------------
            for canonical in canonical_to_build:
                converted_values = []
                selected = None

                for actual in present_aliases[canonical]:
                    raw_value = row[index[actual.upper()]]

                    if raw_value is None or normalize_text(raw_value) is None:
                        continue

                    try:
                        converted = convert_canonical_value(canonical, raw_value)
                    except ValueError:
                        issue_book.add(
                            source_path,
                            "DATA ISSUE",
                            f"INVALID_{canonical}_TYPE",
                            f"{canonical} contains a value that cannot be converted "
                            f"to its canonical type.",
                            sample_oid=oid,
                        )
                        continue

                    if converted is not None:
                        converted_values.append(converted)
                        if selected is None:
                            selected = converted

                # Multiple source aliases disagree.
                if len(set(converted_values)) > 1:
                    canonical_conflicts.add(canonical)
                    issue_book.add(
                        source_path,
                        "DATA ISSUE",
                        "FIELD_VALUE_CONFLICT",
                        f"Multiple source aliases mapped to {canonical} contain "
                        f"different normalized values.",
                        sample_oid=oid,
                    )

                values[canonical] = selected

            # Missing LAST_UPD_D field gets the first-cycle baseline.
            if not source_has_last_upd_d:
                values["LAST_UPD_D"] = BASELINE_LAST_UPD_D

            # Capture original source IDs before lookup filling.
            source_branch = values.get("BRANCH_CODE")
            source_landoffice = values.get("LANDOFFICE")

            if source_branch is not None:
                source_branch_values.add(source_branch)
            if source_landoffice is not None:
                source_landoffice_values.add(source_landoffice)

            # ---------------------------------------------------------
            # BRANCH_CODE / LANDOFFICE resolution
            # ---------------------------------------------------------
            branch = source_branch
            landoffice = source_landoffice

            if branch is not None:
                if not valid_branch_code(branch):
                    issue_book.add(
                        source_path,
                        "ROUTING BLOCKER",
                        "INVALID_BRANCH_CODE",
                        "Source BRANCH_CODE is present but is not exactly 8 digits.",
                        sample_oid=oid,
                    )
                else:
                    if landoffice is not None:
                        if (branch, landoffice) not in lookup["valid_pairs"]:
                            issue_book.add(
                                source_path,
                                "DATA ISSUE",
                                "BRANCH_LANDOFFICE_CONFLICT",
                                "Source BRANCH_CODE / LANDOFFICE pair does not exist "
                                "in OldBranch. Source values are preserved.",
                                sample_oid=oid,
                            )
                    else:
                        candidates = lookup["branch_to_landoffices"].get(branch, set())

                        if len(candidates) == 1:
                            landoffice = next(iter(candidates))
                        elif len(candidates) == 0:
                            issue_book.add(
                                source_path,
                                "DATA ISSUE",
                                "LANDOFFICE_NOT_FOUND",
                                "BRANCH_CODE is valid for routing, but no LANDOFFICE "
                                "was found in OldBranch.",
                                sample_oid=oid,
                            )
                        else:
                            issue_book.add(
                                source_path,
                                "DATA ISSUE",
                                "LANDOFFICE_AMBIGUOUS_FOR_BRANCH",
                                "BRANCH_CODE is valid for routing, but OldBranch "
                                "contains multiple LANDOFFICE values for the branch.",
                                sample_oid=oid,
                            )

            else:
                if landoffice is None:
                    issue_book.add(
                        source_path,
                        "ROUTING BLOCKER",
                        "BRANCH_AND_LANDOFFICE_MISSING",
                        "Neither BRANCH_CODE nor LANDOFFICE can be resolved.",
                        sample_oid=oid,
                    )
                else:
                    candidates = lookup["landoffice_to_branches"].get(landoffice, set())

                    if len(candidates) == 1:
                        branch = next(iter(candidates))
                    elif len(candidates) == 0:
                        issue_book.add(
                            source_path,
                            "ROUTING BLOCKER",
                            "BRANCH_NOT_FOUND_FROM_LANDOFFICE",
                            "LANDOFFICE has no BRANCH_CODE in OldBranch.",
                            sample_oid=oid,
                        )
                    else:
                        issue_book.add(
                            source_path,
                            "ROUTING BLOCKER",
                            "AMBIGUOUS_BRANCH_FROM_LANDOFFICE",
                            "LANDOFFICE maps to multiple historical BRANCH_CODE "
                            "values. STATUS=Active is not used to guess history.",
                            sample_oid=oid,
                        )

            values["BRANCH_CODE"] = branch
            values["LANDOFFICE"] = landoffice

            if branch is not None:
                resolved_branch_values.add(branch)
            if landoffice is not None:
                resolved_landoffice_values.add(landoffice)

            # ---------------------------------------------------------
            # LAST_UPD_D validation
            # ---------------------------------------------------------
            if source_has_last_upd_d:
                last_upd = values.get("LAST_UPD_D")
                if last_upd is None:
                    issue_book.add(
                        source_path,
                        "WARNING",
                        "LAST_UPD_D_NULL",
                        "LAST_UPD_D field exists but this row is NULL; source NULL "
                        "is preserved.",
                        sample_oid=oid,
                    )
                elif not valid_last_upd_d(last_upd):
                    issue_book.add(
                        source_path,
                        "DATA ISSUE",
                        "INVALID_LAST_UPD_D",
                        "LAST_UPD_D is not a valid YYYYMMDDHHMMSS timestamp.",
                        sample_oid=oid,
                    )

            # ---------------------------------------------------------
            # Canonical validation (report only; no guessing)
            # ---------------------------------------------------------
            validate_admin_hierarchy(values, issue_book, source_path, oid)
            validate_utm_values(values, issue_book, source_path, oid)

            # ---------------------------------------------------------
            # Row routing
            # ---------------------------------------------------------
            if valid_branch_code(branch):
                branch_counts[branch] += 1
                collect_reference_candidate(reference_candidates, values)
            else:
                blocked_count += 1

            # Write normalized values to temporary canonical fields.
            for canonical, temp_name in temp_dest_fields.items():
                row[index[temp_name.upper()]] = values.get(canonical)

            cursor.updateRow(row)

    if not source_has_last_upd_d:
        issue_book.add(
            source_path,
            "WARNING",
            "BASELINE_LAST_UPD_D_APPLIED",
            f"Source has no LAST_UPD_D field. Baseline {BASELINE_LAST_UPD_D} "
            "was applied.",
            count=source_count,
        )

    # -------------------------------------------------------------
    # Replace aliases with canonical fields
    # -------------------------------------------------------------
    for canonical in canonical_to_build:
        aliases = present_aliases[canonical]

        if canonical in canonical_conflicts:
            # Preserve all conflicting aliases. If the original field already
            # uses the canonical name, rename it to SRC_<canonical> first so
            # the normalized destination can take the canonical name.
            for actual in aliases:
                if actual.upper() == canonical.upper():
                    preserved_name = unique_field_name(
                        temp_fc,
                        f"SRC_{canonical}",
                    )
                    arcpy.management.AlterField(
                        str(temp_fc),
                        actual,
                        preserved_name,
                        preserved_name,
                    )
        else:
            # Equivalent/redundant aliases can be collapsed into the normalized
            # canonical field because their source meaning is preserved.
            delete_fields = [
                actual
                for actual in aliases
                if get_actual_field(temp_fc, actual) is not None
            ]
            if delete_fields:
                arcpy.management.DeleteField(str(temp_fc), delete_fields)

        temp_name = temp_dest_fields[canonical]
        arcpy.management.AlterField(
            str(temp_fc),
            temp_name,
            canonical,
            canonical,
        )

    return {
        "branch_counts": branch_counts,
        "blocked_count": blocked_count,
        "source_branch_values": source_branch_values,
        "source_landoffice_values": source_landoffice_values,
        "resolved_branch_values": resolved_branch_values,
        "resolved_landoffice_values": resolved_landoffice_values,
    }


def ensure_target_schema(target_fc, source_fc, source_path, issue_book):
    """
    Add source fields missing from the target FC.

    Canonical fields use contract types/lengths.
    Unmapped source text fields are created as TEXT(255) to prevent truncation
    when later shapefiles contain longer values.
    """
    target_fields = field_map(target_fc)

    for source_field in arcpy.ListFields(str(source_fc)):
        if is_system_field(source_field):
            continue

        source_name = source_field.name
        target_field = target_fields.get(source_name.upper())

        if target_field is None:
            canonical_spec = CANONICAL_SPECS.get(source_name.upper())

            if canonical_spec is not None:
                add_field_from_spec(target_fc, source_name, canonical_spec)
            else:
                add_type = arc_field_type_for_add(source_field)

                if add_type is None:
                    issue_book.add(
                        source_path,
                        "DATA ISSUE",
                        "UNSUPPORTED_SOURCE_FIELD_TYPE",
                        f"Field {source_name} has unsupported ArcGIS type "
                        f"{source_field.type}; it could not be added to target.",
                    )
                    continue

                kwargs = {}
                if add_type == "TEXT":
                    kwargs["field_length"] = max(source_field.length or 1, 255)

                arcpy.management.AddField(
                    str(target_fc),
                    source_name,
                    add_type,
                    **kwargs,
                )

            target_fields = field_map(target_fc)
            continue

        # Report incompatible same-name schemas instead of silently pretending
        # they are identical.
        canonical_spec = CANONICAL_SPECS.get(source_name.upper())

        if canonical_spec is not None:
            expected_type = expected_arc_field_class(canonical_spec)
            if target_field.type != expected_type:
                issue_book.add(
                    source_path,
                    "DATA ISSUE",
                    "TARGET_FIELD_TYPE_CONFLICT",
                    f"Target field {source_name} type {target_field.type} "
                    f"does not match canonical type {expected_type}.",
                )
        elif target_field.type != source_field.type:
            issue_book.add(
                source_path,
                "DATA ISSUE",
                "SOURCE_FIELD_TYPE_CONFLICT",
                f"Unmapped field {source_name} has source type "
                f"{source_field.type} but target type {target_field.type}.",
            )


def get_or_create_target_fc(
    branch_code,
    zone,
    wkid,
    source_fc,
    source_path,
    issue_book,
):
    target_name = f"A{branch_code}_{zone}"
    target_fc = PREPARE_GDB / target_name

    if not arcpy.Exists(str(target_fc)):
        sr = arcpy.SpatialReference(wkid)
        arcpy.management.CreateFeatureclass(
            str(PREPARE_GDB),
            target_name,
            "POLYGON",
            spatial_reference=sr,
        )
    else:
        target_wkid = dataset_wkid(target_fc)
        if target_wkid != wkid:
            raise RuntimeError(
                f"Target {target_name} has WKID {target_wkid}, expected {wkid}."
            )

    ensure_target_schema(target_fc, source_fc, source_path, issue_book)
    return target_fc


def append_branch_rows(
    source_fc,
    branch_code,
    zone,
    wkid,
    source_path,
    source_signature_value,
    issue_book,
):
    target_fc = get_or_create_target_fc(
        branch_code,
        zone,
        wkid,
        source_fc,
        source_path,
        issue_book,
    )

    # Persistent checkpoint is written BEFORE any rows are appended.
    # If execution is interrupted, the next run removes rows with OID values
    # greater than this checkpoint and retries the source shapefile.
    checkpoint_target_before_append(
        source_path=source_path,
        source_signature_value=source_signature_value,
        target_fc=target_fc,
    )

    field_sql = arcpy.AddFieldDelimiters(str(source_fc), "BRANCH_CODE")
    safe_branch = branch_code.replace("'", "''")
    where_clause = f"{field_sql} = '{safe_branch}'"

    layer_name = f"lyr_{uuid.uuid4().hex[:10]}"

    try:
        arcpy.management.MakeFeatureLayer(
            str(source_fc),
            layer_name,
            where_clause,
        )

        expected_count = int(arcpy.management.GetCount(layer_name)[0])
        before_count = int(arcpy.management.GetCount(str(target_fc))[0])

        arcpy.management.Append(
            layer_name,
            str(target_fc),
            "NO_TEST",
        )

        after_count = int(arcpy.management.GetCount(str(target_fc))[0])
        actual_count = after_count - before_count

        if actual_count != expected_count:
            issue_book.add(
                source_path,
                "DATA ISSUE",
                "APPEND_COUNT_MISMATCH",
                f"{target_fc.name}: selected {expected_count} rows but "
                f"target count increased by {actual_count}.",
                count=abs(expected_count - actual_count) or 1,
            )

        return target_fc.name, actual_count

    finally:
        if arcpy.Exists(layer_name):
            arcpy.management.Delete(layer_name)


In [ ]:
# ---------------------------------------------------------------------
# Reference tables and Excel report
# ---------------------------------------------------------------------

def normalize_reference_key(value):
    if value is None:
        return None
    if isinstance(value, str):
        return normalize_text(value)
    return value


def ensure_reference_table(table_name, code_field, name_field):
    table_path = REFERENCE_GDB / table_name

    if not arcpy.Exists(str(table_path)):
        arcpy.management.CreateTable(str(REFERENCE_GDB), table_name)

        code_spec = CANONICAL_SPECS[code_field]
        add_field_from_spec(table_path, code_field, code_spec)

        # Names are staging/reference text, intentionally generous.
        arcpy.management.AddField(
            str(table_path),
            name_field,
            "TEXT",
            field_length=255,
        )

    return table_path


def update_reference_tables(reference_candidates, issue_book):
    inserted = Counter()

    for table_name, (code_field, name_field) in REFERENCE_DEFINITIONS.items():
        table_path = ensure_reference_table(table_name, code_field, name_field)

        existing = defaultdict(set)
        with arcpy.da.SearchCursor(
            str(table_path),
            [code_field, name_field],
        ) as cursor:
            for code, name in cursor:
                code = normalize_reference_key(code)
                name = normalize_text(name)
                if code is not None and name is not None:
                    existing[code].add(name)

        candidate_map = defaultdict(set)
        for code, name in reference_candidates[table_name]:
            code = normalize_reference_key(code)
            name = normalize_text(name)
            if code is not None and name is not None:
                candidate_map[code].add(name)

        rows_to_insert = []

        for code, names in candidate_map.items():
            # More than one semantic name in this acquisition: do not choose.
            if len(names) > 1:
                issue_book.add(
                    REFERENCE_GDB,
                    "DATA ISSUE",
                    f"{table_name}_NAME_CONFLICT",
                    f"{code_field}={code!r} maps to multiple {name_field} "
                    f"values in this acquisition: {sorted(names)!r}",
                )
                continue

            candidate_name = next(iter(names))

            if code in existing:
                if candidate_name not in existing[code]:
                    issue_book.add(
                        REFERENCE_GDB,
                        "DATA ISSUE",
                        f"{table_name}_REFERENCE_CONFLICT",
                        f"{code_field}={code!r} already maps to "
                        f"{sorted(existing[code])!r}; new value "
                        f"{candidate_name!r} was not inserted.",
                    )
                continue

            rows_to_insert.append((code, candidate_name))

        if rows_to_insert:
            with arcpy.da.InsertCursor(
                str(table_path),
                [code_field, name_field],
            ) as cursor:
                for row in sorted(rows_to_insert, key=lambda x: str(x[0])):
                    cursor.insertRow(row)
                    inserted[table_name] += 1

    return inserted


def create_report_table(table_name, schema, rows):
    table_path = WORK_GDB / table_name

    if arcpy.Exists(str(table_path)):
        arcpy.management.Delete(str(table_path))

    arcpy.management.CreateTable(str(WORK_GDB), table_name)

    for field_name, field_type, field_length in schema:
        kwargs = {}
        if field_type == "TEXT":
            kwargs["field_length"] = field_length
        arcpy.management.AddField(
            str(table_path),
            field_name,
            field_type,
            **kwargs,
        )

    field_names = [field_name for field_name, _, _ in schema]

    if rows:
        with arcpy.da.InsertCursor(str(table_path), field_names) as cursor:
            for record in rows:
                cursor.insertRow([record.get(name) for name in field_names])

    return table_path


FILES_SCHEMA = [
    ("SOURCE_PATH", "TEXT", 1000),
    ("SOURCE_NAME", "TEXT", 255),
    ("SOURCE_FEATURE_COUNT", "LONG", None),
    ("GEOMETRY_TYPE", "TEXT", 30),
    ("WKID", "LONG", None),
    ("ZONE", "TEXT", 2),
    ("SOURCE_BRANCH_CODE", "TEXT", 1000),
    ("SOURCE_LANDOFFICE", "TEXT", 1000),
    ("RESOLVED_BRANCH_CODE", "TEXT", 1000),
    ("RESOLVED_LANDOFFICE", "TEXT", 1000),
    ("TARGET_FEATURE_CLASS", "TEXT", 2000),
    ("IMPORTED_FEATURE_COUNT", "LONG", None),
    ("BLOCKED_FEATURE_COUNT", "LONG", None),
    ("STATUS", "TEXT", 40),
    ("ISSUE_COUNT", "LONG", None),
]

FIELDS_SCHEMA = [
    ("SOURCE_PATH", "TEXT", 1000),
    ("SOURCE_FIELD", "TEXT", 128),
    ("SOURCE_TYPE", "TEXT", 40),
    ("SOURCE_LENGTH", "LONG", None),
    ("CANONICAL_FIELD", "TEXT", 128),
    ("ACTION", "TEXT", 40),
]

ISSUES_SCHEMA = [
    ("SOURCE_PATH", "TEXT", 1000),
    ("SEVERITY", "TEXT", 40),
    ("ISSUE_CODE", "TEXT", 100),
    ("MESSAGE", "TEXT", 2000),
    ("ISSUE_COUNT", "LONG", None),
    ("SAMPLE_OID", "TEXT", 64),
]

SUMMARY_SCHEMA = [
    ("CATEGORY", "TEXT", 50),
    ("KEY", "TEXT", 1000),
    ("COUNT", "LONG", None),
]


def build_summary_rows(file_rows, issue_rows, target_counts, branch_counts):
    summary_rows = []

    def add_counter(category, counter):
        for key, count in sorted(counter.items(), key=lambda x: str(x[0])):
            summary_rows.append({
                "CATEGORY": category,
                "KEY": str(key),
                "COUNT": int(count),
            })

    add_counter(
        "STATUS",
        Counter(row["STATUS"] for row in file_rows),
    )

    add_counter(
        "ZONE_FILES",
        Counter(
            row["ZONE"]
            for row in file_rows
            if row["ZONE"] not in (None, "")
        ),
    )

    add_counter(
        "ISSUE_TYPE",
        Counter({
            code: sum(
                row["ISSUE_COUNT"]
                for row in issue_rows
                if row["ISSUE_CODE"] == code
            )
            for code in {row["ISSUE_CODE"] for row in issue_rows}
        }),
    )

    add_counter("TARGET_FEATURE_CLASS", target_counts)
    add_counter("BRANCH_IMPORTED_ROWS", branch_counts)

    return summary_rows


def write_excel_report(file_rows, field_rows, issue_rows, summary_rows):
    tables = [
        create_report_table("FILES", FILES_SCHEMA, file_rows),
        create_report_table("FIELDS", FIELDS_SCHEMA, field_rows),
        create_report_table("ISSUES", ISSUES_SCHEMA, issue_rows),
        create_report_table("SUMMARY", SUMMARY_SCHEMA, summary_rows),
    ]

    if ACQUIRED_REPORT.exists():
        ACQUIRED_REPORT.unlink()

    # ArcGIS TableToExcel supports multiple input tables; each table becomes
    # its own worksheet named from the input table.
    arcpy.conversion.TableToExcel(
        [str(table) for table in tables],
        str(ACQUIRED_REPORT),
    )


In [ ]:
# ---------------------------------------------------------------------
# Persistent resume / checkpoint control
# ---------------------------------------------------------------------

IMPORT_STATE = PREPARE_GDB / "IMPORT_STATE"
IMPORT_TARGET_STATE = PREPARE_GDB / "IMPORT_TARGET_STATE"
IMPORT_FILE_REPORT = PREPARE_GDB / "IMPORT_FILE_REPORT"
IMPORT_ISSUES = PREPARE_GDB / "IMPORT_ISSUES"
IMPORT_REF_CANDIDATES = PREPARE_GDB / "IMPORT_REF_CANDIDATES"


def now_text():
    return dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S")


def source_signature(source_path):
    """
    Fingerprint a shapefile dataset from its common sidecar files.

    RAW is supposed to be immutable, but the signature protects resume mode
    from silently skipping a source that was replaced after a previous run.
    """
    source_path = Path(source_path)
    base = source_path.with_suffix("")

    parts = []
    for suffix in [".shp", ".dbf", ".shx", ".prj", ".cpg", ".sbn", ".sbx"]:
        part = base.with_suffix(suffix)
        if part.exists():
            stat = part.stat()
            parts.append(
                f"{part.name.lower()}|{stat.st_size}|{stat.st_mtime_ns}"
            )

    payload = "\n".join(parts).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()


def ensure_table(table_path, schema):
    if arcpy.Exists(str(table_path)):
        return table_path

    arcpy.management.CreateTable(
        str(Path(table_path).parent),
        Path(table_path).name,
    )

    for field_name, field_type, field_length in schema:
        kwargs = {}
        if field_type == "TEXT":
            kwargs["field_length"] = field_length
        arcpy.management.AddField(
            str(table_path),
            field_name,
            field_type,
            **kwargs,
        )

    return table_path


IMPORT_STATE_SCHEMA = [
    ("SOURCE_PATH", "TEXT", 1000),
    ("SOURCE_SIGNATURE", "TEXT", 64),
    ("STATUS", "TEXT", 30),
    ("STARTED_AT", "TEXT", 19),
    ("COMPLETED_AT", "TEXT", 19),
    ("MESSAGE", "TEXT", 1000),
]

IMPORT_TARGET_STATE_SCHEMA = [
    ("SOURCE_PATH", "TEXT", 1000),
    ("SOURCE_SIGNATURE", "TEXT", 64),
    ("TARGET_FC", "TEXT", 255),
    ("BEFORE_MAX_OID", "LONG", None),
    ("BEFORE_COUNT", "LONG", None),
]

IMPORT_FILE_REPORT_SCHEMA = [
    ("SOURCE_SIGNATURE", "TEXT", 64),
] + FILES_SCHEMA

IMPORT_ISSUES_SCHEMA = [
    ("SOURCE_SIGNATURE", "TEXT", 64),
] + ISSUES_SCHEMA

IMPORT_REF_CANDIDATES_SCHEMA = [
    ("SOURCE_PATH", "TEXT", 1000),
    ("SOURCE_SIGNATURE", "TEXT", 64),
    ("TABLE_NAME", "TEXT", 30),
    ("CODE_VALUE", "TEXT", 100),
    ("NAME_VALUE", "TEXT", 500),
]


def ensure_resume_tables():
    ensure_table(IMPORT_STATE, IMPORT_STATE_SCHEMA)
    ensure_table(IMPORT_TARGET_STATE, IMPORT_TARGET_STATE_SCHEMA)
    ensure_table(IMPORT_FILE_REPORT, IMPORT_FILE_REPORT_SCHEMA)
    ensure_table(IMPORT_ISSUES, IMPORT_ISSUES_SCHEMA)
    ensure_table(IMPORT_REF_CANDIDATES, IMPORT_REF_CANDIDATES_SCHEMA)


def sql_text_equals(dataset, field_name, value):
    field_sql = arcpy.AddFieldDelimiters(str(dataset), field_name)
    safe = str(value).replace("'", "''")
    return f"{field_sql} = '{safe}'"


def delete_rows_for_source(table_path, source_path):
    where = sql_text_equals(table_path, "SOURCE_PATH", str(source_path))
    with arcpy.da.UpdateCursor(str(table_path), ["SOURCE_PATH"], where) as cursor:
        for _ in cursor:
            cursor.deleteRow()


def get_import_state(source_path):
    where = sql_text_equals(IMPORT_STATE, "SOURCE_PATH", str(source_path))

    with arcpy.da.SearchCursor(
        str(IMPORT_STATE),
        ["SOURCE_SIGNATURE", "STATUS", "STARTED_AT", "COMPLETED_AT", "MESSAGE"],
        where,
    ) as cursor:
        rows = list(cursor)

    if not rows:
        return None

    sig, status, started, completed, message = rows[-1]
    return {
        "SOURCE_SIGNATURE": sig,
        "STATUS": status,
        "STARTED_AT": started,
        "COMPLETED_AT": completed,
        "MESSAGE": message,
    }


def mark_source_started(source_path, source_signature_value):
    delete_rows_for_source(IMPORT_STATE, source_path)
    delete_rows_for_source(IMPORT_TARGET_STATE, source_path)

    with arcpy.da.InsertCursor(
        str(IMPORT_STATE),
        [
            "SOURCE_PATH",
            "SOURCE_SIGNATURE",
            "STATUS",
            "STARTED_AT",
            "COMPLETED_AT",
            "MESSAGE",
        ],
    ) as cursor:
        cursor.insertRow([
            str(source_path),
            source_signature_value,
            "IN_PROGRESS",
            now_text(),
            None,
            None,
        ])


def mark_source_complete(source_path, source_signature_value, status, message=None):
    where = sql_text_equals(IMPORT_STATE, "SOURCE_PATH", str(source_path))

    updated = False
    with arcpy.da.UpdateCursor(
        str(IMPORT_STATE),
        [
            "SOURCE_SIGNATURE",
            "STATUS",
            "COMPLETED_AT",
            "MESSAGE",
        ],
        where,
    ) as cursor:
        for row in cursor:
            row[0] = source_signature_value
            row[1] = status
            row[2] = now_text()
            row[3] = message
            cursor.updateRow(row)
            updated = True

    if not updated:
        mark_source_started(source_path, source_signature_value)
        mark_source_complete(
            source_path,
            source_signature_value,
            status,
            message,
        )


def max_oid_and_count(feature_class):
    count = int(arcpy.management.GetCount(str(feature_class))[0])

    if count == 0:
        return 0, 0

    oid_field = arcpy.Describe(str(feature_class)).OIDFieldName
    max_oid = 0

    with arcpy.da.SearchCursor(str(feature_class), [oid_field]) as cursor:
        for (oid,) in cursor:
            if oid is not None and oid > max_oid:
                max_oid = oid

    return max_oid, count


def checkpoint_target_before_append(
    source_path,
    source_signature_value,
    target_fc,
):
    """
    Record the target state exactly once for this source/target pair,
    immediately before the first append.
    """
    where_source = sql_text_equals(
        IMPORT_TARGET_STATE,
        "SOURCE_PATH",
        str(source_path),
    )

    existing_targets = set()
    with arcpy.da.SearchCursor(
        str(IMPORT_TARGET_STATE),
        ["TARGET_FC"],
        where_source,
    ) as cursor:
        for (target_name,) in cursor:
            existing_targets.add(target_name)

    target_name = Path(target_fc).name
    if target_name in existing_targets:
        return

    before_max_oid, before_count = max_oid_and_count(target_fc)

    with arcpy.da.InsertCursor(
        str(IMPORT_TARGET_STATE),
        [
            "SOURCE_PATH",
            "SOURCE_SIGNATURE",
            "TARGET_FC",
            "BEFORE_MAX_OID",
            "BEFORE_COUNT",
        ],
    ) as cursor:
        cursor.insertRow([
            str(source_path),
            source_signature_value,
            target_name,
            before_max_oid,
            before_count,
        ])


def rollback_incomplete_source(source_path):
    """
    Restore every target touched by an incomplete source to its pre-append
    checkpoint. This assumes this notebook is the only writer to PREPARE_GDB.
    """
    where = sql_text_equals(
        IMPORT_TARGET_STATE,
        "SOURCE_PATH",
        str(source_path),
    )

    checkpoints = []
    with arcpy.da.SearchCursor(
        str(IMPORT_TARGET_STATE),
        ["TARGET_FC", "BEFORE_MAX_OID", "BEFORE_COUNT"],
        where,
    ) as cursor:
        checkpoints.extend(cursor)

    for target_name, before_max_oid, before_count in checkpoints:
        target_fc = PREPARE_GDB / target_name

        if not arcpy.Exists(str(target_fc)):
            if int(before_count or 0) != 0:
                raise RuntimeError(
                    f"Cannot recover {source_path}: expected existing target "
                    f"{target_name} with {before_count} rows."
                )
            continue

        oid_field = arcpy.Describe(str(target_fc)).OIDFieldName
        oid_sql = arcpy.AddFieldDelimiters(str(target_fc), oid_field)
        where_new = f"{oid_sql} > {int(before_max_oid or 0)}"

        deleted = 0
        with arcpy.da.UpdateCursor(
            str(target_fc),
            [oid_field],
            where_new,
        ) as cursor:
            for _ in cursor:
                cursor.deleteRow()
                deleted += 1

        after_count = int(arcpy.management.GetCount(str(target_fc))[0])

        if after_count != int(before_count or 0):
            raise RuntimeError(
                f"Resume recovery failed for {target_name}. "
                f"Expected {before_count} rows after rollback, found {after_count}. "
                "Do not continue automatically; check for external edits."
            )

        print(
            f"Recovered incomplete source: {Path(source_path).name} | "
            f"{target_name} | removed {deleted:,} partial row(s)"
        )

    # Old report/candidate rows for the incomplete source are discarded.
    delete_rows_for_source(IMPORT_FILE_REPORT, source_path)
    delete_rows_for_source(IMPORT_ISSUES, source_path)
    delete_rows_for_source(IMPORT_REF_CANDIDATES, source_path)
    delete_rows_for_source(IMPORT_TARGET_STATE, source_path)
    delete_rows_for_source(IMPORT_STATE, source_path)


def recover_all_incomplete_sources():
    in_progress = []

    with arcpy.da.SearchCursor(
        str(IMPORT_STATE),
        ["SOURCE_PATH", "STATUS"],
    ) as cursor:
        for source_path, status in cursor:
            if status == "IN_PROGRESS":
                in_progress.append(source_path)

    for source_path in in_progress:
        rollback_incomplete_source(source_path)

    return len(in_progress)


def should_skip_completed(source_path, source_signature_value):
    if not RESUME_IMPORT:
        return False

    state = get_import_state(source_path)
    if not state:
        return False

    if (
        state["STATUS"] in {"COMPLETE", "BLOCKED"}
        and state["SOURCE_SIGNATURE"] == source_signature_value
    ):
        return True

    if (
        state["STATUS"] in {"COMPLETE", "BLOCKED"}
        and state["SOURCE_SIGNATURE"] != source_signature_value
    ):
        raise RuntimeError(
            f"RAW source changed after previous import: {source_path}\n"
            "The source signature no longer matches the completed checkpoint. "
            "Because RAW must be immutable, review this file before continuing."
        )

    return False


def persist_file_report_row(record, source_signature_value):
    source_path = record["SOURCE_PATH"]
    delete_rows_for_source(IMPORT_FILE_REPORT, source_path)

    fields = ["SOURCE_SIGNATURE"] + [name for name, _, _ in FILES_SCHEMA]

    with arcpy.da.InsertCursor(str(IMPORT_FILE_REPORT), fields) as cursor:
        cursor.insertRow(
            [source_signature_value]
            + [record.get(name) for name, _, _ in FILES_SCHEMA]
        )


def persist_source_issues(source_path, source_signature_value, issue_book):
    delete_rows_for_source(IMPORT_ISSUES, source_path)

    rows = [
        row for row in issue_book.rows()
        if row["SOURCE_PATH"] == str(source_path)
    ]

    fields = ["SOURCE_SIGNATURE"] + [name for name, _, _ in ISSUES_SCHEMA]

    if rows:
        with arcpy.da.InsertCursor(str(IMPORT_ISSUES), fields) as cursor:
            for record in rows:
                cursor.insertRow(
                    [source_signature_value]
                    + [record.get(name) for name, _, _ in ISSUES_SCHEMA]
                )


def persist_reference_candidates(
    source_path,
    source_signature_value,
    source_candidates,
):
    delete_rows_for_source(IMPORT_REF_CANDIDATES, source_path)

    fields = [
        "SOURCE_PATH",
        "SOURCE_SIGNATURE",
        "TABLE_NAME",
        "CODE_VALUE",
        "NAME_VALUE",
    ]

    with arcpy.da.InsertCursor(
        str(IMPORT_REF_CANDIDATES),
        fields,
    ) as cursor:
        for table_name, pairs in source_candidates.items():
            for code, name in sorted(pairs, key=lambda x: (str(x[0]), str(x[1]))):
                cursor.insertRow([
                    str(source_path),
                    source_signature_value,
                    table_name,
                    str(code),
                    normalize_text(name),
                ])


def load_persistent_file_rows():
    fields = [name for name, _, _ in FILES_SCHEMA]
    rows = []

    with arcpy.da.SearchCursor(
        str(IMPORT_FILE_REPORT),
        fields,
    ) as cursor:
        for values in cursor:
            rows.append(dict(zip(fields, values)))

    return rows


def load_persistent_issues(global_issue_book=None):
    fields = [name for name, _, _ in ISSUES_SCHEMA]
    rows = []

    with arcpy.da.SearchCursor(
        str(IMPORT_ISSUES),
        fields,
    ) as cursor:
        for values in cursor:
            rows.append(dict(zip(fields, values)))

    if global_issue_book is not None:
        # Keep non-source/global issues calculated in this run as well.
        persistent_paths = {
            row["SOURCE_PATH"]
            for row in rows
        }

        for row in global_issue_book.rows():
            if row["SOURCE_PATH"] not in persistent_paths:
                rows.append(row)

    return rows


def load_persistent_reference_candidates():
    result = {
        table_name: set()
        for table_name in REFERENCE_DEFINITIONS
    }

    with arcpy.da.SearchCursor(
        str(IMPORT_REF_CANDIDATES),
        ["TABLE_NAME", "CODE_VALUE", "NAME_VALUE"],
    ) as cursor:
        for table_name, code_value, name_value in cursor:
            if table_name not in result:
                continue

            code_field, _ = REFERENCE_DEFINITIONS[table_name]
            field_type, _ = CANONICAL_SPECS[code_field]

            if field_type == "LONG":
                try:
                    code = to_long(code_value)
                except ValueError:
                    continue
            else:
                code = normalize_text(code_value)

            name = normalize_text(name_value)
            if code is not None and name is not None:
                result[table_name].add((code, name))

    return result

In [ ]:
# ---------------------------------------------------------------------
# Main first-import execution
# ---------------------------------------------------------------------

def run_first_import():
    issues = IssueBook()
    field_rows = []

    target_import_counts = Counter()
    branch_import_counts = Counter()

    setup_workspaces()
    ensure_resume_tables()

    recovered_count = recover_all_incomplete_sources()
    if recovered_count:
        print(f"Recovered {recovered_count:,} incomplete source file(s).")

    lookup = load_oldbranch_lookup(issues)
    shapefiles = discover_shapefiles()

    print(f"Discovered {len(shapefiles):,} shapefile(s).")

    skipped_count = 0
    processed_count = 0

    for file_no, source_path in enumerate(shapefiles, start=1):
        source_path = Path(source_path)
        sig = source_signature(source_path)

        # Field inventory is cheap and is regenerated every run so the final
        # FIELDS worksheet remains complete even when feature imports are skipped.
        field_rows.extend(field_inventory_rows(source_path))

        if should_skip_completed(source_path, sig):
            skipped_count += 1
            print(
                f"[{file_no:,}/{len(shapefiles):,}] SKIP COMPLETE | "
                f"{source_path}"
            )
            continue

        processed_count += 1

        source_count = 0
        geometry_type = None
        wkid = 0
        zone = None

        imported_count = 0
        blocked_count = 0

        source_branch_values = set()
        source_landoffice_values = set()
        resolved_branch_values = set()
        resolved_landoffice_values = set()

        target_names = []

        # Candidate collection is per-source so it can be checkpointed.
        source_reference_candidates = {
            table_name: set()
            for table_name in REFERENCE_DEFINITIONS
        }

        print(f"[{file_no:,}/{len(shapefiles):,}] PROCESS | {source_path}")

        # Remove stale issue/candidate/report rows before the retry starts.
        delete_rows_for_source(IMPORT_FILE_REPORT, source_path)
        delete_rows_for_source(IMPORT_ISSUES, source_path)
        delete_rows_for_source(IMPORT_REF_CANDIDATES, source_path)

        mark_source_started(source_path, sig)

        try:
            source_count = int(arcpy.management.GetCount(str(source_path))[0])
            geometry_type = dataset_geometry_type(source_path)
            wkid = dataset_wkid(source_path)
            zone = SUPPORTED_WKID.get(wkid)

            # ---------------------------------------------------------
            # File-level routing blockers
            # ---------------------------------------------------------
            if geometry_type.lower() != "polygon":
                blocked_count = source_count
                issues.add(
                    source_path,
                    "ROUTING BLOCKER",
                    "UNSUPPORTED_GEOMETRY",
                    f"Geometry type is {geometry_type!r}; only Polygon is accepted.",
                    count=max(source_count, 1),
                )

            elif wkid not in SUPPORTED_WKID:
                blocked_count = source_count
                issues.add(
                    source_path,
                    "ROUTING BLOCKER",
                    "UNSUPPORTED_WKID",
                    f"WKID={wkid}; only 24047 and 24048 are accepted.",
                    count=max(source_count, 1),
                )

            else:
                # -----------------------------------------------------
                # Copy to temporary GDB; RAW remains untouched.
                # -----------------------------------------------------
                temp_name = f"SRC_{file_no:04d}_{uuid.uuid4().hex[:8]}"
                temp_fc = WORK_GDB / temp_name

                try:
                    arcpy.management.CopyFeatures(
                        str(source_path),
                        str(temp_fc),
                    )

                    stats = normalize_staging_feature_class(
                        temp_fc=temp_fc,
                        source_path=source_path,
                        source_count=source_count,
                        lookup=lookup,
                        issue_book=issues,
                        reference_candidates=source_reference_candidates,
                    )

                    blocked_count = stats["blocked_count"]
                    source_branch_values = stats["source_branch_values"]
                    source_landoffice_values = stats["source_landoffice_values"]
                    resolved_branch_values = stats["resolved_branch_values"]
                    resolved_landoffice_values = stats["resolved_landoffice_values"]

                    # ---------------------------------------------
                    # Row-level routing into A<BRANCH_CODE>_<ZONE>
                    # ---------------------------------------------
                    for branch_code, expected_rows in sorted(
                        stats["branch_counts"].items()
                    ):
                        target_name, appended_rows = append_branch_rows(
                            source_fc=temp_fc,
                            branch_code=branch_code,
                            zone=zone,
                            wkid=wkid,
                            source_path=source_path,
                            source_signature_value=sig,
                            issue_book=issues,
                        )

                        target_names.append(target_name)
                        imported_count += appended_rows
                        target_import_counts[target_name] += appended_rows
                        branch_import_counts[branch_code] += appended_rows

                        if appended_rows != expected_rows:
                            issues.add(
                                source_path,
                                "DATA ISSUE",
                                "ROUTING_COUNT_MISMATCH",
                                f"BRANCH_CODE={branch_code}: normalization identified "
                                f"{expected_rows} routable rows but {appended_rows} "
                                "rows were appended.",
                                count=abs(expected_rows - appended_rows) or 1,
                            )

                finally:
                    if "temp_fc" in locals() and arcpy.Exists(str(temp_fc)):
                        arcpy.management.Delete(str(temp_fc))

            # ---------------------------------------------------------
            # Reconciliation
            # ---------------------------------------------------------
            if source_count != imported_count + blocked_count:
                issues.add(
                    source_path,
                    "DATA ISSUE",
                    "SOURCE_IMPORT_RECONCILIATION_FAILED",
                    f"source={source_count}, imported={imported_count}, "
                    f"blocked={blocked_count}; values do not reconcile.",
                    count=abs(
                        source_count - imported_count - blocked_count
                    ) or 1,
                )

            issue_count = issues.count_for(source_path)

            if imported_count == 0 and blocked_count == source_count:
                status = "BLOCKED"
            elif issue_count:
                status = "IMPORTED_WITH_ISSUES"
            else:
                status = "IMPORTED"

            file_record = {
                "SOURCE_PATH": str(source_path),
                "SOURCE_NAME": source_path.name,
                "SOURCE_FEATURE_COUNT": source_count,
                "GEOMETRY_TYPE": geometry_type,
                "WKID": wkid,
                "ZONE": zone,
                "SOURCE_BRANCH_CODE": preview_values(source_branch_values),
                "SOURCE_LANDOFFICE": preview_values(source_landoffice_values),
                "RESOLVED_BRANCH_CODE": preview_values(resolved_branch_values),
                "RESOLVED_LANDOFFICE": preview_values(resolved_landoffice_values),
                "TARGET_FEATURE_CLASS": ", ".join(
                    sorted(set(target_names))
                ),
                "IMPORTED_FEATURE_COUNT": imported_count,
                "BLOCKED_FEATURE_COUNT": blocked_count,
                "STATUS": status,
                "ISSUE_COUNT": issue_count,
            }

            # Persist all metadata BEFORE marking the source complete.
            persist_file_report_row(file_record, sig)
            persist_source_issues(source_path, sig, issues)
            persist_reference_candidates(
                source_path,
                sig,
                source_reference_candidates,
            )

            state_status = (
                "BLOCKED"
                if status == "BLOCKED"
                else "COMPLETE"
            )
            mark_source_complete(
                source_path,
                sig,
                state_status,
                message=status,
            )

            # Once complete, rollback checkpoints for this source are no longer
            # needed. The source-level COMPLETE checkpoint is retained.
            delete_rows_for_source(
                IMPORT_TARGET_STATE,
                source_path,
            )

        except KeyboardInterrupt:
            # Deliberately leave source state as IN_PROGRESS.
            # The next run will rollback only this source and retry it.
            print()
            print(
                "Import interrupted by user. Completed shapefiles are safe. "
                "The current shapefile remains IN_PROGRESS and will be "
                "rolled back/retried next run."
            )
            raise

        except Exception as exc:
            # Keep IN_PROGRESS so the next run performs rollback before retry.
            issues.add(
                source_path,
                "ROUTING BLOCKER",
                "PROCESSING_EXCEPTION",
                f"{type(exc).__name__}: {exc}",
                count=max(source_count, 1),
            )
            persist_source_issues(source_path, sig, issues)

            print("  ERROR:", exc)
            traceback.print_exc()

            # Stop here rather than moving on after a potentially partial append.
            # This guarantees a single well-defined recovery point.
            raise

    # -----------------------------------------------------------------
    # Build final outputs from PERSISTENT checkpoints, including files
    # completed during previous notebook runs.
    # -----------------------------------------------------------------
    file_rows = load_persistent_file_rows()
    reference_candidates = load_persistent_reference_candidates()

    reference_inserted = update_reference_tables(
        reference_candidates,
        issues,
    )

    issue_rows = load_persistent_issues(global_issue_book=issues)

    # Reconstruct total counts from persistent FILES report.
    total_target_counts = Counter()
    total_branch_counts = Counter()

    for row in file_rows:
        count = int(row.get("IMPORTED_FEATURE_COUNT") or 0)
        targets = [
            item.strip()
            for item in str(row.get("TARGET_FEATURE_CLASS") or "").split(",")
            if item.strip()
        ]

        # A file can be row-routed to several targets. The exact per-target
        # count is not stored in FILES, so use current PREPARE_GDB feature
        # counts for the target summary after the whole cycle is complete.
        for target in targets:
            if arcpy.Exists(str(PREPARE_GDB / target)):
                total_target_counts[target] = int(
                    arcpy.management.GetCount(
                        str(PREPARE_GDB / target)
                    )[0]
                )

        for branch_text in str(
            row.get("RESOLVED_BRANCH_CODE") or ""
        ).split(","):
            branch = branch_text.strip()
            if valid_branch_code(branch) and len(targets) == 1:
                total_branch_counts[branch] += count

    summary_rows = build_summary_rows(
        file_rows=file_rows,
        issue_rows=issue_rows,
        target_counts=total_target_counts,
        branch_counts=total_branch_counts,
    )

    write_excel_report(
        file_rows=file_rows,
        field_rows=field_rows,
        issue_rows=issue_rows,
        summary_rows=summary_rows,
    )

    completed_states = Counter()
    with arcpy.da.SearchCursor(
        str(IMPORT_STATE),
        ["STATUS"],
    ) as cursor:
        for (status,) in cursor:
            completed_states[status] += 1

    print()
    print("FIRST IMPORT PASS COMPLETE")
    print("--------------------------")
    print(f"Source shapefiles : {len(shapefiles):,}")
    print(f"Processed this run: {processed_count:,}")
    print(f"Skipped complete : {skipped_count:,}")
    print(f"Checkpoint states: {dict(completed_states)}")
    print(f"PREPARE_GDB      : {PREPARE_GDB}")
    print(f"Import report    : {ACQUIRED_REPORT}")
    print(f"Reference inserts: {dict(reference_inserted)}")

    if not KEEP_WORK_GDB and arcpy.Exists(str(WORK_GDB)):
        arcpy.management.Delete(str(WORK_GDB))

    return {
        "PREPARE_GDB": str(PREPARE_GDB),
        "ACQUIRED_REPORT": str(ACQUIRED_REPORT),
        "REFERENCE_GDB": str(REFERENCE_GDB),
        "SOURCE_FILES": len(shapefiles),
        "PROCESSED_THIS_RUN": processed_count,
        "SKIPPED_COMPLETE": skipped_count,
        "CHECKPOINT_STATES": dict(completed_states),
        "ISSUE_GROUPS": len(issue_rows),
    }


RESULT = run_first_import()
RESULT

## Expected outputs and resume behavior

For `ACQUIRED = "2022-05"`:

```text
D:\A02-Projects\Refine\01_acquired\A2022-05.gdb
    ├── A<BRANCH_CODE>_47
    ├── A<BRANCH_CODE>_48
    ├── IMPORT_STATE
    ├── IMPORT_TARGET_STATE
    ├── IMPORT_FILE_REPORT
    ├── IMPORT_ISSUES
    └── IMPORT_REF_CANDIDATES
```

The `IMPORT_*` tables are small technical control tables used only to make a large import resumable and auditable. They do not add fields to parcel features.

### Normal restart

Keep:

```python
REBUILD_PREPARE_GDB = False
RESUME_IMPORT = True
```

Then rerun the notebook.

Files already marked `COMPLETE` or `BLOCKED` with the same RAW source signature are skipped.

### If execution is stopped during a shapefile

The active source remains:

```text
STATUS = IN_PROGRESS
```

On the next run, the notebook:

1. reads the target checkpoints written before append,
2. deletes only rows appended by that incomplete source,
3. verifies each target returned to its pre-import row count,
4. clears the incomplete checkpoint,
5. retries that source,
6. continues with the remaining shapefiles.

This prevents already completed source shapefiles from being deleted or re-imported.

**Assumption:** while the notebook is running/resuming, `PREPARE_GDB` must not be edited by another process. Recovery deliberately stops if the expected pre-import target count cannot be restored.

### Intentional full rebuild

Only when you explicitly want to discard all staging work:

```python
REBUILD_PREPARE_GDB = True
```

After that rebuild, set it back to `False`.

### LAST_UPD_D

The source shapefile is allowed to have no `LAST_UPD_D` field.

```text
field exists
    → preserve and validate source values

field absent
    → create canonical LAST_UPD_D
    → fill 20220531000000
```

An existing `LAST_UPD_D` field containing NULL remains a source NULL; the baseline is applied when the **field itself is absent**.

### Import report

```text
D:\A02-Projects\Refine\05_reports\2022-05\01_2022-05-Import.xlsx
```

Worksheets:

- `FILES`
- `FIELDS`
- `ISSUES`
- `SUMMARY`

Persistent report/checkpoint metadata means a final resumed run includes shapefiles completed during earlier notebook runs.

### Reconciliation rule

For every processed shapefile:

```text
SOURCE_FEATURE_COUNT
=
IMPORTED_FEATURE_COUNT
+
BLOCKED_FEATURE_COUNT
```

Any unexplained difference is recorded as `SOURCE_IMPORT_RECONCILIATION_FAILED`.

### Later stages — intentionally not included

This notebook still stops after acquired/staging import and reference/report generation. It does not create APID, hashes, entity/version identifiers, or archive status.